# 프로젝트 1 - Weekend 2: RAG 파이프라인

| 항목 | 내용 |
|------|------|
| **프로젝트** | 주택청약 FAQ 챗봇 - 벡터 검색 업그레이드 |

FAQ 챗봇에 벡터 검색을 업그레이드 해주세요

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w2_token_embedding_vector_rag_project/p1_weekend2_rag_pipeline_0321_%E1%84%80%E1%85%B5%E1%86%B7%E1%84%86%E1%85%B5%E1%86%AB%E1%84%8B%E1%85%A1.ipynb)

## 환경 설정

In [1]:
# 필요한 패키지 설치
# faiss-cpu: Facebook이 만든 벡터 유사도 검색 라이브러리 (GPU 없이 사용하는 CPU 버전)
# langchain-community: FAISS 등 커뮤니티 통합 모듈
!pip install -q openai langchain-openai langchain-community faiss-cpu python-dotenv gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [3]:
# Colab 환경 설정
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# ── 주요 라이브러리 임포트 ────────────────────────────────────────
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # OpenAIEmbeddings: 텍스트→벡터 변환기
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS  # FAISS: 벡터 유사도 검색 엔진

# ── 클라이언트 초기화 ─────────────────────────────────────────────
client = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ⭐ Weekend 2의 핵심: 임베딩 모델
# text-embedding-3-small: OpenAI의 경량 임베딩 모델 (1536차원 벡터 생성)
# 텍스트를 숫자 벡터로 변환하여 의미적 유사도 비교를 가능하게 함
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ 환경 설정 완료")

✅ 환경 설정 완료


## 📦 실습용 샘플 데이터

In [4]:
# 주택청약 FAQ 샘플 데이터 (실습용)
# Weekend 1과 동일한 데이터를 사용하되,
# 이번에는 키워드 매칭이 아닌 벡터 검색으로 업그레이드합니다.
SAMPLE_FAQ_DATA = [
    {"id": "FAQ001", "category": "청약통장",
     "question": "주택청약종합저축이란 무엇인가요?",
     "answer": "주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨",
     "keywords": ["청약종합저축", "만능통장", "납입", "가입"], "difficulty": "easy"},
    {"id": "FAQ004", "category": "청약통장",
     "question": "청약통장 1순위 조건은 무엇인가요?",
     "answer": "1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입",
     "keywords": ["1순위", "가입기간", "예치금", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ005", "category": "청약자격",
     "question": "주택 청약 신청 자격 조건은 무엇인가요?",
     "answer": "1) 만 19세 이상 (기혼자는 연령 제한 없음)\n2) 청약통장 가입 필수\n3) 국민주택: 무주택 세대구성원\n4) 민영주택: 세대주 또는 세대원 가능\n※ 투기과열지구는 세대주만 청약 가능",
     "keywords": ["청약자격", "만19세", "무주택", "세대주"], "difficulty": "easy"},
    {"id": "FAQ006", "category": "청약자격",
     "question": "무주택자 기준은 무엇인가요?",
     "answer": "본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함",
     "keywords": ["무주택", "세대원", "소형주택", "분양권"], "difficulty": "medium"},
    {"id": "FAQ009", "category": "특별공급",
     "question": "특별공급의 종류에는 어떤 것이 있나요?",
     "answer": "1) 기관추천 (국가유공자, 장애인 등)\n2) 다자녀가구 (3명 이상)\n3) 신혼부부 (혼인 7년 이내)\n4) 생애최초 (최초 주택 구입)\n5) 노부모부양 (만 65세 이상 부모)\n※ 2021년부터 신혼/생애최초 물량 확대",
     "keywords": ["특별공급", "기관추천", "다자녀", "신혼부부", "생애최초"], "difficulty": "medium"},
    {"id": "FAQ010", "category": "특별공급",
     "question": "신혼부부 특별공급 조건은 무엇인가요?",
     "answer": "1) 혼인기간 7년 이내 무주택 세대주\n2) 소득: 도시근로자 월평균소득 100~140%\n3) 전용면적 85㎡ 이하\n4) 혼인기간 짧을수록 + 자녀 많을수록 가점 높음\n5) 예비 신혼부부도 신청 가능",
     "keywords": ["신혼부부", "혼인기간", "소득기준", "가점"], "difficulty": "medium"},
    {"id": "FAQ013", "category": "일반공급",
     "question": "가점제와 추첨제의 차이는 무엇인가요?",
     "answer": "가점제: 무주택기간+부양가족+가입기간으로 점수화 (84점 만점)\n추첨제: 무작위 추첨\n1) 투기과열지구: 가점제 100%\n2) 청약과열지역: 가점 75% + 추첨 25%\n3) 기타: 가점 40% + 추첨 60%",
     "keywords": ["가점제", "추첨제", "84점", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ017", "category": "당첨/계약",
     "question": "당첨자 발표는 어떻게 확인하나요?",
     "answer": "1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인",
     "keywords": ["당첨자발표", "청약홈", "SMS알림", "서류제출"], "difficulty": "easy"},
    {"id": "FAQ020", "category": "당첨/계약",
     "question": "재당첨 제한이란 무엇인가요?",
     "answer": "당첨 후 일정 기간 다른 주택 청약 불가:\n1) 투기과열지구: 10년\n2) 청약과열지역: 7년\n3) 수도권 공공주택: 5년\n※ 세대원 전원 적용 (배우자 당첨 시 본인도 제한)",
     "keywords": ["재당첨제한", "10년", "7년", "세대원"], "difficulty": "medium"},
    {"id": "FAQ023", "category": "기타",
     "question": "청약홈 사이트는 어떻게 이용하나요?",
     "answer": "청약홈(www.applyhome.co.kr) - 한국부동산원 운영\n1) 회원가입 후 공인인증서/간편인증 로그인\n2) 청약 신청, 당첨 확인, 가점 계산 가능\n3) 모바일 앱(청약홈)도 동일 서비스 제공",
     "keywords": ["청약홈", "공인인증서", "간편인증", "가점계산"], "difficulty": "easy"},
]

SAMPLE_TEST_QUERIES = [
    {"query": "청약통장 가입하려면 어떻게 해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ001"},
    {"query": "1순위 되려면 뭐가 필요해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ004"},
    {"query": "신혼부부 특공 자격이 궁금해요", "expected_category": "특별공급", "expected_faq_id": "FAQ010"},
    {"query": "가점이 높으면 유리한가요?", "expected_category": "일반공급", "expected_faq_id": "FAQ013"},
    {"query": "당첨되면 어떻게 확인해요?", "expected_category": "당첨/계약", "expected_faq_id": "FAQ017"},
]

print(f"📦 FAQ 데이터 로드 완료: {len(SAMPLE_FAQ_DATA)}개 QA, {len(SAMPLE_TEST_QUERIES)}개 테스트 질의")

📦 FAQ 데이터 로드 완료: 10개 QA, 5개 테스트 질의


---
## 사이클 1: Weekend 1 복원 + Document 객체

FAQ 데이터를 LangChain `Document` 객체 리스트로 변환하세요. `page_content`에 질문+답변, `metadata`에 id/category를 넣으세요.

In [5]:
# ── 사이클 1: Weekend 1 복원 + Document 객체 ──────────────────────
# ⭐ Document: LangChain에서 텍스트를 다루는 기본 단위
#
# Document 구조:
#   page_content (str)  : 실제 텍스트 내용 (검색/임베딩 대상)
#   metadata     (dict) : 부가 정보 (출처, 카테고리 등)
#
# Weekend 1에서는 FAQ 딕셔너리를 직접 다뤘지만,
# Weekend 2에서는 Document 객체로 변환하여 LangChain 생태계와 통합합니다.
# → FAISS, Retriever 등이 모두 Document 형식을 기대하기 때문!

from langchain_core.documents import Document

# FAQ 데이터 → Document 리스트 변환
documents = []
for faq in SAMPLE_FAQ_DATA:
    # page_content: 질문과 답변을 합쳐서 하나의 텍스트로 구성
    # → 질문만 넣으면 답변 내용으로 검색이 안 되고,
    #   답변만 넣으면 질문 형태의 쿼리와 매칭이 안 됨
    #   둘 다 넣어야 검색 정확도가 높아짐!
    content = f"질문: {faq['question']}\n답변: {faq['answer']}"

    # metadata: 검색 후 출처 추적, 필터링에 사용
    metadata = {
        "id": faq["id"],
        "category": faq["category"],
        "difficulty": faq["difficulty"]
    }

    documents.append(Document(page_content=content, metadata=metadata))

# 결과 확인
print(f"📄 Document 개수: {len(documents)}개")
print(f"\n📋 첫 번째 Document 예시:")
print(f"  page_content: {documents[0].page_content[:80]}...")
print(f"  metadata: {documents[0].metadata}")

📄 Document 개수: 10개

📋 첫 번째 Document 예시:
  page_content: 질문: 주택청약종합저축이란 무엇인가요?
답변: 주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.
1) 매월 2만원~...
  metadata: {'id': 'FAQ001', 'category': '청약통장', 'difficulty': 'easy'}


---
## 사이클 2: OpenAI Embeddings

`OpenAIEmbeddings`로 FAQ 텍스트들을 임베딩하고, 질문 간 코사인 유사도를 계산하세요. 의미적으로 유사한 질문이 높은 유사도를 보이는지 확인하세요.

In [6]:
# ── 사이클 2: OpenAI Embeddings ───────────────────────────────────
# ⭐ 임베딩(Embedding)이란?
#   텍스트를 고정 길이의 숫자 벡터(리스트)로 변환하는 것
#   예: "청약통장" → [0.012, -0.034, 0.056, ..., 0.078]  (1536차원)
#
# 왜 벡터로 바꿀까?
#   - 텍스트는 컴퓨터가 직접 비교하기 어려움
#   - 벡터로 바꾸면 "거리"로 의미적 유사도를 수치화할 수 있음
#   - "청약통장 가입" ↔ "주택청약저축 개설" → 키워드는 다르지만 벡터는 가까움!
#
# 코사인 유사도(Cosine Similarity):
#   두 벡터가 얼마나 같은 방향을 가리키는지 측정 (-1 ~ 1)
#   1에 가까울수록 의미가 유사, 0이면 무관, -1이면 정반대

import numpy as np
from langchain_openai import OpenAIEmbeddings

# ── Step 1: 텍스트를 벡터로 변환 ─────────────────────────────────
# embed_documents(): 여러 텍스트를 한 번에 임베딩 (배치 처리)
# embed_query(): 검색 쿼리 하나를 임베딩
texts = [faq["question"] for faq in SAMPLE_FAQ_DATA]
vectors = embeddings.embed_documents(texts)

print(f"📐 임베딩 차원: {len(vectors[0])}차원")
print(f"📄 임베딩된 텍스트 수: {len(vectors)}개")
print(f"🔢 첫 벡터 앞 5개 값: {vectors[0][:5]}")

# ── Step 2: 코사인 유사도 계산 ───────────────────────────────────
def cosine_similarity(a, b):
    """두 벡터 간 코사인 유사도를 계산합니다.
    공식: cos(θ) = (A·B) / (|A| × |B|)
    """
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# ── Step 3: 유사도 비교 실험 ─────────────────────────────────────
# 의미적으로 가까운 질문 vs 먼 질문의 유사도 차이를 확인
test_pairs = [
    ("청약통장 가입하려면 어떻게 해요?", "주택청약종합저축이란 무엇인가요?"),  # 유사
    ("청약통장 가입하려면 어떻게 해요?", "가점제와 추첨제의 차이는 무엇인가요?"),  # 다소 다름
    ("신혼부부 특별공급 조건은 무엇인가요?", "재당첨 제한이란 무엇인가요?"),  # 다름
]

print("\n📊 질문 쌍별 코사인 유사도:")
for q1, q2 in test_pairs:
    v1 = embeddings.embed_query(q1)
    v2 = embeddings.embed_query(q2)
    sim = cosine_similarity(v1, v2)
    print(f"  {sim:.4f} | \"{q1[:20]}...\" ↔ \"{q2[:20]}...\"")

print("\n💡 의미적으로 가까운 질문일수록 유사도가 높음을 확인!")

📐 임베딩 차원: 1536차원
📄 임베딩된 텍스트 수: 10개
🔢 첫 벡터 앞 5개 값: [0.022369384765625, 0.03387451171875, 0.058441162109375, 0.060638427734375, -0.019134521484375]

📊 질문 쌍별 코사인 유사도:
  0.3334 | "청약통장 가입하려면 어떻게 해요?..." ↔ "주택청약종합저축이란 무엇인가요?..."
  0.1342 | "청약통장 가입하려면 어떻게 해요?..." ↔ "가점제와 추첨제의 차이는 무엇인가요?..."
  0.1635 | "신혼부부 특별공급 조건은 무엇인가요?..." ↔ "재당첨 제한이란 무엇인가요?..."

💡 의미적으로 가까운 질문일수록 유사도가 높음을 확인!


---
## 사이클 3: FAISS 벡터 스토어

`FAISS.from_documents()`로 벡터 스토어를 만들고, `similarity_search()`와 `similarity_search_with_score()`로 검색하세요. 저장/로드도 테스트하세요.

In [7]:
# ── 사이클 3: FAISS 벡터 스토어 ───────────────────────────────────
# ⭐ FAISS (Facebook AI Similarity Search):
#   Facebook(Meta)이 개발한 고속 벡터 유사도 검색 라이브러리
#   수백만 개의 벡터에서도 빠르게 가장 유사한 벡터를 찾아줌
#
# 벡터 스토어의 역할:
#   1. Document의 page_content를 임베딩하여 벡터로 변환
#   2. 변환된 벡터들을 인덱스에 저장
#   3. 검색 쿼리가 들어오면 → 쿼리도 임베딩 → 가장 가까운 벡터 찾기
#
# Weekend 1의 키워드 검색과의 차이:
#   키워드 검색: "청약통장" 이라는 단어가 있어야 검색됨
#   벡터 검색 : "저축 계좌 개설" 같은 유사 표현도 검색됨!

from langchain_community.vectorstores import FAISS

# ── Step 1: 벡터 스토어 생성 ─────────────────────────────────────
# from_documents(): Document 리스트 + 임베딩 모델 → 벡터 스토어 생성
# 내부적으로: 각 Document의 page_content → 임베딩 → FAISS 인덱스에 저장
vectorstore = FAISS.from_documents(documents, embeddings)
print(f"✅ 벡터 스토어 생성 완료! 저장된 벡터 수: {vectorstore.index.ntotal}개")

# ── Step 2: similarity_search() ──────────────────────────────────
# 쿼리와 가장 유사한 Document k개를 반환 (기본 k=4)
query = "청약통장 가입 방법이 궁금해요"
results = vectorstore.similarity_search(query, k=3)

print(f"\n🔍 검색 쿼리: \"{query}\"")
for i, doc in enumerate(results, 1):
    print(f"  [{i}] [{doc.metadata['id']}] {doc.metadata['category']}")
    print(f"      {doc.page_content[:60]}...")

# ── Step 3: similarity_search_with_score() ───────────────────────
# 유사도 점수(L2 거리)도 함께 반환
# ⭐ 점수가 낮을수록 더 유사! (L2 거리 = 유클리드 거리)
results_with_score = vectorstore.similarity_search_with_score(query, k=3)

print(f"\n📊 유사도 점수 (L2 거리 - 낮을수록 유사):")
for doc, score in results_with_score:
    print(f"  {score:.4f} | [{doc.metadata['id']}] {doc.metadata['category']}")

# ── Step 4: 저장 & 로드 ─────────────────────────────────────────
# save_local(): 벡터 인덱스를 디스크에 저장 (재시작 시 재임베딩 불필요)
# load_local(): 저장된 인덱스를 불러옴
vectorstore.save_local("faiss_faq_index")
print("\n💾 벡터 스토어 저장 완료: faiss_faq_index/")

# allow_dangerous_deserialization: pickle 역직렬화 허용 (신뢰할 수 있는 파일만!)
loaded_vs = FAISS.load_local("faiss_faq_index", embeddings, allow_dangerous_deserialization=True)
print(f"📂 로드 완료! 벡터 수: {loaded_vs.index.ntotal}개")

✅ 벡터 스토어 생성 완료! 저장된 벡터 수: 10개

🔍 검색 쿼리: "청약통장 가입 방법이 궁금해요"
  [1] [FAQ017] 당첨/계약
      질문: 당첨자 발표는 어떻게 확인하나요?
답변: 1) 청약홈(www.applyhome.co.kr) 접속
2)...
  [2] [FAQ023] 기타
      질문: 청약홈 사이트는 어떻게 이용하나요?
답변: 청약홈(www.applyhome.co.kr) - 한국부동산...
  [3] [FAQ005] 청약자격
      질문: 주택 청약 신청 자격 조건은 무엇인가요?
답변: 1) 만 19세 이상 (기혼자는 연령 제한 없음)
2...

📊 유사도 점수 (L2 거리 - 낮을수록 유사):
  1.0800 | [FAQ017] 당첨/계약
  1.0926 | [FAQ023] 기타
  1.2589 | [FAQ005] 청약자격

💾 벡터 스토어 저장 완료: faiss_faq_index/
📂 로드 완료! 벡터 수: 10개


---
## 사이클 4: Retriever 구성

벡터 스토어에서 `as_retriever()`로 retriever를 만들고, `similarity` vs `mmr` 검색 타입과 `k=1,3,5` 결과를 비교하세요.

In [8]:
# ── 사이클 4: Retriever 구성 ──────────────────────────────────────
# ⭐ Retriever: 검색 기능을 LCEL 체인에 연결할 수 있게 해주는 인터페이스
#
# vectorstore.similarity_search()를 직접 호출해도 되지만,
# Retriever로 감싸면 LCEL 파이프(|)로 체인에 연결 가능!
#
# 검색 타입 비교:
#   similarity: 단순 유사도 순서로 k개 반환 (가장 기본적)
#   mmr (Maximal Marginal Relevance):
#     유사도 높은 것 + 결과 간 다양성까지 고려
#     → 비슷한 내용이 중복으로 나오는 것을 방지!

# ── Step 1: 기본 Retriever 생성 ──────────────────────────────────
# as_retriever(): 벡터 스토어를 Retriever 인터페이스로 변환
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Retriever는 .invoke()로 호출 → Document 리스트 반환
query = "신혼부부 청약 자격이 궁금해요"
docs = retriever.invoke(query)

print(f"🔍 기본 Retriever (similarity, k=3):")
for doc in docs:
    print(f"  [{doc.metadata['id']}] {doc.metadata['category']} - {doc.page_content[:50]}...")

# ── Step 2: k값에 따른 결과 비교 ─────────────────────────────────
print(f"\n📊 k값별 검색 결과 비교 (쿼리: \"{query[:20]}...\"):")
for k in [1, 3, 5]:
    ret = vectorstore.as_retriever(search_kwargs={"k": k})
    results = ret.invoke(query)
    ids = [d.metadata["id"] for d in results]
    print(f"  k={k}: {ids}")

# ── Step 3: similarity vs MMR 비교 ───────────────────────────────
# MMR은 search_type 파라미터로 지정
# fetch_k: MMR이 후보로 가져올 문서 수 (이 중에서 다양성 고려하여 k개 선택)
print(f"\n📊 검색 타입 비교:")

ret_sim = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
ret_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 6}  # 6개 후보 중 다양성 고려하여 3개 선택
)

sim_docs = ret_sim.invoke(query)
mmr_docs = ret_mmr.invoke(query)

print(f"  similarity: {[d.metadata['id'] for d in sim_docs]}")
print(f"       카테고리: {[d.metadata['category'] for d in sim_docs]}")
print(f"  mmr:        {[d.metadata['id'] for d in mmr_docs]}")
print(f"       카테고리: {[d.metadata['category'] for d in mmr_docs]}")
print("\n💡 MMR은 다양한 카테고리의 결과를 가져오는 경향이 있음!")

🔍 기본 Retriever (similarity, k=3):
  [FAQ010] 특별공급 - 질문: 신혼부부 특별공급 조건은 무엇인가요?
답변: 1) 혼인기간 7년 이내 무주택 세대주...
  [FAQ005] 청약자격 - 질문: 주택 청약 신청 자격 조건은 무엇인가요?
답변: 1) 만 19세 이상 (기혼자는 연...
  [FAQ017] 당첨/계약 - 질문: 당첨자 발표는 어떻게 확인하나요?
답변: 1) 청약홈(www.applyhome.co...

📊 k값별 검색 결과 비교 (쿼리: "신혼부부 청약 자격이 궁금해요..."):
  k=1: ['FAQ010']
  k=3: ['FAQ010', 'FAQ005', 'FAQ017']
  k=5: ['FAQ010', 'FAQ005', 'FAQ017', 'FAQ023', 'FAQ009']

📊 검색 타입 비교:
  similarity: ['FAQ010', 'FAQ005', 'FAQ017']
       카테고리: ['특별공급', '청약자격', '당첨/계약']
  mmr:        ['FAQ010', 'FAQ023', 'FAQ004']
       카테고리: ['특별공급', '기타', '청약통장']

💡 MMR은 다양한 카테고리의 결과를 가져오는 경향이 있음!


---
## 사이클 5: RAG 체인

`retriever | format_docs`를 context로 사용하는 RAG 체인을 LCEL로 만들고, 질문 5개로 테스트하세요. Weekend 1의 키워드 검색 대비 답변 품질 차이를 확인하세요.

In [9]:
# ── 사이클 5: RAG 체인 ────────────────────────────────────────────
# ⭐ RAG (Retrieval-Augmented Generation) 복습:
#   1. Retrieve: 질문과 관련된 문서를 벡터 DB에서 검색
#   2. Augment:  검색된 문서를 프롬프트에 삽입
#   3. Generate: LLM이 문서를 참고하여 답변 생성
#
# Weekend 1과의 차이:
#   W1: search_faq() → 키워드 매칭 → 정확한 단어가 있어야 검색됨
#   W2: retriever   → 벡터 유사도 → 의미적으로 비슷하면 검색됨!
#
# LCEL 체인 구조:
#   {"context": retriever | format_docs, "question": RunnablePassthrough()}
#     → prompt → llm → StrOutputParser()

from langchain_core.runnables import RunnablePassthrough

# ── Step 1: format_docs 함수 ─────────────────────────────────────
# Retriever가 반환한 Document 리스트를 하나의 문자열로 합침
# → LLM 프롬프트에 삽입할 수 있는 형태로 변환
def format_docs(docs):
    """Document 리스트 → 프롬프트용 문자열로 변환"""
    return "\n---\n".join(
        f"[{d.metadata.get('category', '')}] {d.page_content}"
        for d in docs
    )

# ── Step 2: RAG 프롬프트 템플릿 ──────────────────────────────────
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 주택청약 전문 상담원입니다. "
     "아래 참고 문서를 기반으로 정확하고 친절하게 답변해주세요.\n\n"
     "참고 문서:\n{context}\n\n"
     "참고 문서에 없는 내용은 '해당 정보는 제공된 자료에 없습니다'라고 안내하세요."),
    ("user", "{question}")
])

# ── Step 3: RAG 체인 조립 (LCEL) ─────────────────────────────────
# RunnablePassthrough(): 입력을 그대로 통과시키는 유틸리티
# 체인 흐름:
#   질문(str) → {"context": retriever→format_docs 결과, "question": 원래 질문}
#            → prompt → llm → StrOutputParser() → 답변(str)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# ── Step 4: 테스트 ───────────────────────────────────────────────
test_questions = [
    "청약통장 가입하려면 어떻게 해요?",
    "1순위 되려면 뭐가 필요해요?",
    "신혼부부 특공 자격이 궁금해요",
    "가점이 높으면 유리한가요?",
    "당첨되면 어떻게 확인해요?",
]

print("🔍 RAG 체인 테스트 (벡터 검색 기반):\n")
for q in test_questions:
    answer = rag_chain.invoke(q)
    print(f"❓ {q}")
    print(f"💬 {answer[:150]}...")
    print("-" * 60)

print("\n💡 Weekend 1의 키워드 검색과 비교해보세요!")
print("   → '저축 계좌', '배우자 청약' 같은 유사 표현으로도 검색 가능")

🔍 RAG 체인 테스트 (벡터 검색 기반):

❓ 청약통장 가입하려면 어떻게 해요?
💬 해당 정보는 제공된 자료에 없습니다....
------------------------------------------------------------
❓ 1순위 되려면 뭐가 필요해요?
💬 1순위 조건은 주택 유형에 따라 다릅니다.

1) 민영주택: 수도권에서는 12개월 이상, 비수도권에서는 6개월 이상 예치금이 필요합니다.
2) 국민주택: 수도권에서는 12개월(24회) 이상, 비수도권에서는 6개월(12회) 이상 예치금이 필요합니다.
3) 투기과열지구: ...
------------------------------------------------------------
❓ 신혼부부 특공 자격이 궁금해요
💬 신혼부부 특별공급의 자격 조건은 다음과 같습니다:

1) 혼인기간 7년 이내의 무주택 세대주
2) 소득: 도시근로자 월평균소득의 100%에서 140% 사이
3) 전용면적 85㎡ 이하의 주택
4) 혼인기간이 짧을수록, 자녀가 많을수록 가점이 높습니다.
5) 예비 신혼부부...
------------------------------------------------------------
❓ 가점이 높으면 유리한가요?
💬 네, 가점이 높으면 청약에 유리합니다. 가점제는 무주택기간, 부양가족 수, 가입기간 등을 점수화하여 총 84점 만점으로 평가합니다. 가점이 높을수록 청약 당첨 확률이 높아지기 때문에, 가점이 높은 분들이 유리한 상황입니다. 특히, 투기과열지구에서는 가점제 비율이 100...
------------------------------------------------------------
❓ 당첨되면 어떻게 확인해요?
💬 당첨자 발표는 다음과 같이 확인하실 수 있습니다:

1) 청약홈(www.applyhome.co.kr) 접속
2) 당첨자 조회 메뉴 클릭
3) 문자 알림 서비스 신청 가능

당첨 후에는 서류 제출 기간과 계약 일정도 반드시 확인하시기 바랍니다....
--

---
## 사이클 6: 검색 결과 검증

`SAMPLE_TEST_QUERIES` 5개로 RAG 체인이 올바른 FAQ를 찾는지 검증하세요. `similarity_search_with_score()`로 유사도 점수도 함께 확인하고, 기대한 FAQ ID가 검색 결과에 포함되는지 O/X로 판정하세요. 결과를 표로 정리하세요.

In [10]:
# ── 사이클 6: 검색 결과 검증 ──────────────────────────────────────
# ⭐ 검증(Evaluation)의 중요성:
#   RAG 시스템이 올바른 문서를 찾는지 정량적으로 측정해야
#   "잘 된다"가 아니라 "정확도 80%"처럼 수치로 판단 가능
#
# 검증 방법:
#   1. 테스트 쿼리에 대해 기대하는 FAQ ID가 있음
#   2. 벡터 검색 결과의 상위 k개에 기대 FAQ가 포함되는지 확인
#   3. 유사도 점수(L2 거리)도 함께 기록

print("📊 검색 결과 검증표")
print("=" * 80)
print(f"{'쿼리':<25} {'기대ID':<8} {'검색결과':<25} {'점수':<8} {'판정'}")
print("-" * 80)

hit_count = 0

for tq in SAMPLE_TEST_QUERIES:
    query = tq["query"]
    expected_id = tq["expected_faq_id"]

    # similarity_search_with_score: (Document, L2거리) 튜플 리스트 반환
    results = vectorstore.similarity_search_with_score(query, k=3)

    # 검색된 FAQ ID 목록
    found_ids = [doc.metadata["id"] for doc, score in results]

    # 기대한 FAQ가 검색 결과에 포함되는지 확인
    hit = expected_id in found_ids
    if hit:
        hit_count += 1

    # 가장 유사한 결과의 점수
    top_score = results[0][1] if results else float("inf")
    mark = "⭕" if hit else "❌"

    print(f"{query:<25} {expected_id:<8} {str(found_ids):<25} {top_score:<8.4f} {mark}")

print("-" * 80)
accuracy = hit_count / len(SAMPLE_TEST_QUERIES) * 100
print(f"\n📈 검색 정확도: {hit_count}/{len(SAMPLE_TEST_QUERIES)} ({accuracy:.0f}%)")
print(f"💡 키워드 검색(Weekend 1) 대비 벡터 검색의 정확도 향상 여부를 비교해보세요!")

📊 검색 결과 검증표
쿼리                        기대ID     검색결과                      점수       판정
--------------------------------------------------------------------------------
청약통장 가입하려면 어떻게 해요?        FAQ001   ['FAQ023', 'FAQ017', 'FAQ005'] 1.0086   ❌
1순위 되려면 뭐가 필요해요?          FAQ004   ['FAQ004', 'FAQ013', 'FAQ006'] 1.3491   ⭕
신혼부부 특공 자격이 궁금해요          FAQ010   ['FAQ010', 'FAQ005', 'FAQ017'] 0.9512   ⭕
가점이 높으면 유리한가요?            FAQ013   ['FAQ013', 'FAQ004', 'FAQ006'] 1.4435   ⭕
당첨되면 어떻게 확인해요?            FAQ017   ['FAQ017', 'FAQ023', 'FAQ020'] 1.0363   ⭕
--------------------------------------------------------------------------------

📈 검색 정확도: 4/5 (80%)
💡 키워드 검색(Weekend 1) 대비 벡터 검색의 정확도 향상 여부를 비교해보세요!


---
## 사이클 7: 소스 문서 표시

RAG 답변에 참고한 FAQ 출처(ID, 카테고리)를 함께 반환하는 체인을 만드세요. `RunnableParallel`로 answer와 sources를 동시에 가져오세요.

In [ ]:
# ── 사이클 7: 소스 문서 표시 ──────────────────────────────────────
# ⭐ RunnableParallel: 여러 체인을 동시에(병렬로) 실행
#   → 답변 생성과 출처 추출을 동시에 수행!
#
# 왜 출처가 중요한가?
#   - 답변의 근거를 사용자에게 보여줌 → 신뢰도 ↑
#   - 잘못된 답변 시 어떤 문서를 참고했는지 디버깅 가능
#   - 실서비스에서는 "이 답변은 OO 문서를 참고했습니다" 형태로 표시

from langchain_core.runnables import RunnableParallel

# ── Step 1: 출처 추출 함수 ───────────────────────────────────────
def get_sources(docs):
    """Document 리스트에서 출처 정보(ID, 카테고리)를 추출합니다."""
    return [
        {"id": d.metadata["id"], "category": d.metadata["category"]}
        for d in docs
    ]

# ── Step 2: RunnableParallel로 answer + sources 동시 실행 ────────
# RunnableParallel은 딕셔너리 형태로 여러 체인을 병렬 실행
# 각 키에 대해 독립적인 체인이 동시에 실행되고, 결과가 딕셔너리로 합쳐짐
#
# 흐름:
#   질문(str) ─┬─ "answer"  : retriever | format_docs → prompt → llm → str
#             └─ "sources" : retriever → get_sources → list

rag_chain_with_sources = RunnableParallel(
    answer=(
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | rag_prompt
        | llm
        | StrOutputParser()
    ),
    sources=retriever | get_sources
)

# ── Step 3: 테스트 ───────────────────────────────────────────────
test_queries = [
    "청약통장 1순위 조건이 뭐예요?",
    "신혼부부 특별공급 자격이 궁금해요",
    "당첨 후 제한 사항이 있나요?",
]

for q in test_queries:
    result = rag_chain_with_sources.invoke(q)
    print(f"❓ {q}")
    print(f"💬 {result['answer'][:120]}...")
    print(f"📎 참고 FAQ:")
    for src in result['sources']:
        print(f"   - [{src['id']}] {src['category']}")
    print("-" * 60)

---
## 사이클 8: FAQChatbotV2 클래스

vectorstore, retriever, rag_chain을 하나로 묶는 `FAQChatbotV2` 클래스를 만드세요. `ask(question)` 메서드가 answer, sources, time을 반환하도록 하세요.

In [11]:
# ── 사이클 8: FAQChatbotV2 클래스 ─────────────────────────────────
# ⭐ 지금까지 만든 컴포넌트들을 하나의 클래스로 캡슐화합니다.
#
# 캡슐화의 장점:
#   1. 사용이 간단해짐: chatbot.ask("질문") 한 줄로 모든 것 처리
#   2. 상태 관리: vectorstore, retriever 등을 내부에서 관리
#   3. 재사용성: 다른 곳에서도 쉽게 사용 가능
#
# V1(Weekend 1) vs V2(Weekend 2):
#   V1: 키워드 검색(search_faq) 기반
#   V2: 벡터 검색(FAISS + Retriever) 기반

import time

class FAQChatbotV2:
    """FAISS 벡터 검색 기반 주택청약 FAQ 챗봇 (Weekend 2 버전)"""

    def __init__(self, faq_data, llm, embeddings, k=3):
        """챗봇을 초기화합니다.

        Args:
            faq_data: FAQ 딕셔너리 리스트
            llm: LangChain LLM 객체
            embeddings: 임베딩 모델
            k: 검색할 문서 수
        """
        # Step 1: FAQ → Document 변환
        self.documents = []
        for faq in faq_data:
            content = f"질문: {faq['question']}\n답변: {faq['answer']}"
            metadata = {"id": faq["id"], "category": faq["category"]}
            self.documents.append(Document(page_content=content, metadata=metadata))

        # Step 2: 벡터 스토어 생성
        self.vectorstore = FAISS.from_documents(self.documents, embeddings)

        # Step 3: Retriever 구성
        self.retriever = self.vectorstore.as_retriever(search_kwargs={"k": k})

        # Step 4: RAG 체인 조립
        self.prompt = ChatPromptTemplate.from_messages([
            ("system",
             "당신은 주택청약 전문 상담원입니다. "
             "참고 문서를 기반으로 정확하고 친절하게 답변해주세요.\n\n"
             "참고 문서:\n{context}"),
            ("user", "{question}")
        ])

        self.rag_chain = (
            {"context": self.retriever | self._format_docs,
             "question": RunnablePassthrough()}
            | self.prompt
            | llm
            | StrOutputParser()
        )

        print(f"✅ FAQChatbotV2 초기화 완료 (FAQ {len(self.documents)}개, k={k})")

    def _format_docs(self, docs):
        """Document 리스트를 문자열로 변환 (내부 메서드)"""
        return "\n---\n".join(
            f"[{d.metadata.get('category', '')}] {d.page_content}"
            for d in docs
        )

    def ask(self, question):
        """질문에 답변합니다. answer, sources, time을 반환합니다."""
        # 입력 검증
        if not question or not question.strip():
            return {"answer": "❌ 질문을 입력해주세요.", "sources": [], "time": 0}
        if len(question.strip()) > 500:
            return {"answer": "❌ 500자 이내로 입력해주세요.", "sources": [], "time": 0}

        try:
            start = time.time()

            # 답변 생성
            answer = self.rag_chain.invoke(question)

            # 출처 추출 (별도로 retriever 호출)
            source_docs = self.retriever.invoke(question)
            sources = [
                {"id": d.metadata["id"], "category": d.metadata["category"]}
                for d in source_docs
            ]

            elapsed = round(time.time() - start, 2)
            return {"answer": answer, "sources": sources, "time": elapsed}

        except Exception as e:
            return {"answer": f"❌ 오류 발생: {e}", "sources": [], "time": 0}

# ── 테스트 ────────────────────────────────────────────────────────
chatbot = FAQChatbotV2(SAMPLE_FAQ_DATA, llm, embeddings, k=3)

test_qs = ["청약통장이 뭐예요?", "신혼부부 특공 자격", "당첨 확인 방법"]
for q in test_qs:
    result = chatbot.ask(q)
    print(f"❓ {q}")
    print(f"💬 {result['answer'][:120]}...")
    print(f"📎 출처: {[s['id'] for s in result['sources']]} | ⏱️ {result['time']}초")
    print("-" * 60)

✅ FAQChatbotV2 초기화 완료 (FAQ 10개, k=3)
❓ 청약통장이 뭐예요?
💬 청약통장은 주택청약을 위한 저축 통장으로, 주택을 구매하고자 하는 분들이 청약을 신청할 수 있도록 도와주는 역할을 합니다. 주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장으로, 매월 2만원에서...
📎 출처: ['FAQ023', 'FAQ017', 'FAQ001'] | ⏱️ 2.97초
------------------------------------------------------------
❓ 신혼부부 특공 자격
💬 신혼부부 특별공급의 자격 조건은 다음과 같습니다:

1) 혼인기간이 7년 이내인 무주택 세대주여야 합니다.
2) 소득은 도시근로자 월평균소득의 100%에서 140% 사이여야 합니다.
3) 전용면적은 85㎡ 이하의 주...
📎 출처: ['FAQ010', 'FAQ009', 'FAQ005'] | ⏱️ 4.7초
------------------------------------------------------------
❓ 당첨 확인 방법
💬 당첨자 발표를 확인하는 방법은 다음과 같습니다:

1) 청약홈 웹사이트에 접속합니다: [www.applyhome.co.kr](http://www.applyhome.co.kr)
2) '당첨자 조회' 메뉴를 클릭합니다....
📎 출처: ['FAQ017', 'FAQ023', 'FAQ020'] | ⏱️ 2.27초
------------------------------------------------------------


---
## 사이클 9: Gradio UI v2

`gr.ChatInterface`로 RAG 기반 FAQ 챗봇 UI를 만드세요. 답변에 참고 FAQ 출처도 포함해서 표시하세요.

In [12]:
# ── 사이클 9: Gradio UI v2 ────────────────────────────────────────
# ⭐ gr.ChatInterface: 채팅 UI를 한 줄로 생성하는 Gradio 컴포넌트
#   fn: 메시지를 받아 응답을 반환하는 함수 (message, history) → str
#   history: 이전 대화 기록 (Gradio가 자동 관리)
#
# Weekend 1 UI와의 차이:
#   V1: 키워드 검색 기반 → 정확한 단어 필요
#   V2: 벡터 검색 기반 → 자연어로 자유롭게 질문 가능!

import gradio as gr

def chat_v2(message, history):
    """Gradio ChatInterface용 콜백 함수"""
    result = chatbot.ask(message)

    # 답변 + 출처 + 응답시간을 포맷하여 반환
    response = result["answer"]

    if result["sources"]:
        source_text = ", ".join(
            f"[{s['id']}] {s['category']}" for s in result["sources"]
        )
        response += f"\n\n---\n📎 참고: {source_text}\n⏱️ 응답 시간: {result['time']}초"

    return response

# Gradio UI 생성
demo = gr.ChatInterface(
    fn=chat_v2,
    title="🏠 주택청약 FAQ 챗봇 v2.0 (RAG)",
    description="벡터 검색 기반 주택청약 상담 챗봇입니다. 자연어로 자유롭게 질문하세요!",
    examples=[
        "청약통장이 뭔가요?",
        "1순위 조건 알려주세요",
        "신혼부부 특별공급 자격",
        "가점제와 추첨제 차이",
        "당첨 확인 방법",
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=False, inline=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

---
## 사이클 10: 통합 테스트

전체 RAG 파이프라인을 10개 질문으로 테스트하세요. 각 질문의 응답 시간, 검색된 카테고리, 답변 길이를 포함한 결과표를 출력하고, 전체 정확도와 평균 응답 시간을 요약하세요.

In [13]:
# ── 사이클 10: 통합 테스트 ────────────────────────────────────────
# ⭐ 전체 RAG 파이프라인의 성능을 종합적으로 평가합니다.
#
# 평가 항목:
#   1. 검색 정확도: 기대한 FAQ가 검색 결과에 포함되는지
#   2. 응답 시간: 각 질문의 처리 소요 시간
#   3. 답변 길이: 답변이 적절한 길이인지
#   4. 카테고리 정확성: 올바른 카테고리의 문서를 참조하는지

import time

# 10개 테스트 질문 (기대 결과 포함)
test_set = [
    {"query": "청약통장 가입하려면 어떻게 해요?", "expected_id": "FAQ001", "expected_cat": "청약통장"},
    {"query": "1순위 되려면 뭐가 필요해요?", "expected_id": "FAQ004", "expected_cat": "청약통장"},
    {"query": "청약 신청 자격 조건이 궁금해요", "expected_id": "FAQ005", "expected_cat": "청약자격"},
    {"query": "무주택자는 어떤 기준인가요?", "expected_id": "FAQ006", "expected_cat": "청약자격"},
    {"query": "특별공급 종류가 뭐가 있어요?", "expected_id": "FAQ009", "expected_cat": "특별공급"},
    {"query": "신혼부부 특공 자격이 궁금해요", "expected_id": "FAQ010", "expected_cat": "특별공급"},
    {"query": "가점제가 뭐예요?", "expected_id": "FAQ013", "expected_cat": "일반공급"},
    {"query": "당첨되면 어떻게 확인해요?", "expected_id": "FAQ017", "expected_cat": "당첨/계약"},
    {"query": "재당첨 제한이 뭐예요?", "expected_id": "FAQ020", "expected_cat": "당첨/계약"},
    {"query": "청약홈 사이트 사용법", "expected_id": "FAQ023", "expected_cat": "기타"},
]

print("📊 통합 테스트 결과")
print("=" * 90)
print(f"{'#':<3} {'쿼리':<25} {'검색ID':<22} {'카테고리':<10} {'시간':<6} {'길이':<6} {'판정'}")
print("-" * 90)

total_time = 0
hit_count = 0

for i, tc in enumerate(test_set, 1):
    start = time.time()

    # FAQChatbotV2로 답변 생성
    result = chatbot.ask(tc["query"])
    elapsed = round(time.time() - start, 2)
    total_time += elapsed

    # 검색 정확도 판정
    found_ids = [s["id"] for s in result["sources"]]
    found_cats = [s["category"] for s in result["sources"]]
    hit = tc["expected_id"] in found_ids
    if hit:
        hit_count += 1

    mark = "⭕" if hit else "❌"
    ans_len = len(result["answer"])

    print(f"{i:<3} {tc['query']:<25} {str(found_ids):<22} {found_cats[0] if found_cats else 'N/A':<10} {elapsed:<6} {ans_len:<6} {mark}")

print("-" * 90)

# 요약 통계
accuracy = hit_count / len(test_set) * 100
avg_time = total_time / len(test_set)

print(f"\n📈 종합 결과:")
print(f"  검색 정확도: {hit_count}/{len(test_set)} ({accuracy:.0f}%)")
print(f"  평균 응답 시간: {avg_time:.2f}초")
print(f"  총 소요 시간: {total_time:.2f}초")
print(f"\n💡 Weekend 1(키워드 검색) 대비 Weekend 2(벡터 검색)의 개선 포인트:")
print(f"  - 키워드가 정확히 일치하지 않아도 의미적 검색 가능")
print(f"  - '저축 계좌 개설' → '청약통장' FAQ 검색 가능")
print(f"  - 유사도 점수로 검색 품질을 수치화 가능")

📊 통합 테스트 결과
#   쿼리                        검색ID                   카테고리       시간     길이     판정
------------------------------------------------------------------------------------------
1   청약통장 가입하려면 어떻게 해요?        ['FAQ023', 'FAQ017', 'FAQ005'] 기타         5.36   520    ❌
2   1순위 되려면 뭐가 필요해요?          ['FAQ004', 'FAQ013', 'FAQ006'] 청약통장       12.83  292    ⭕
3   청약 신청 자격 조건이 궁금해요         ['FAQ017', 'FAQ005', 'FAQ023'] 당첨/계약      2.45   202    ⭕
4   무주택자는 어떤 기준인가요?           ['FAQ006', 'FAQ005', 'FAQ010'] 청약자격       3.48   188    ⭕
5   특별공급 종류가 뭐가 있어요?          ['FAQ009', 'FAQ010', 'FAQ004'] 특별공급       3.92   179    ⭕
6   신혼부부 특공 자격이 궁금해요          ['FAQ010', 'FAQ005', 'FAQ017'] 특별공급       3.95   252    ⭕
7   가점제가 뭐예요?                 ['FAQ013', 'FAQ020', 'FAQ023'] 일반공급       1.94   164    ⭕
8   당첨되면 어떻게 확인해요?            ['FAQ017', 'FAQ023', 'FAQ020'] 당첨/계약      1.73   164    ⭕
9   재당첨 제한이 뭐예요?              ['FAQ020', 'FAQ017', 'FAQ004'] 당첨/계약      2.79   187    ⭕
10  청약홈 사이트 사용법         

---
## Weekend 2 완료! 🎉

| 사이클 | 구현 내용 | 핵심 개념 |
|--------|-----------|----------|
| 1 | Document 객체 변환 | `Document(page_content, metadata)` |
| 2 | OpenAI Embeddings | 텍스트→벡터, 코사인 유사도 |
| 3 | FAISS 벡터 스토어 | `from_documents()`, `similarity_search()` |
| 4 | Retriever 구성 | `as_retriever()`, similarity vs mmr |
| 5 | RAG 체인 (LCEL) | `retriever \| format_docs` + `RunnablePassthrough` |
| 6 | 검색 결과 검증 | 정확도 측정, L2 거리 점수 |
| 7 | 소스 문서 표시 | `RunnableParallel` |
| 8 | FAQChatbotV2 클래스 | 캡슐화, ask() 메서드 |
| 9 | Gradio UI v2 | `gr.ChatInterface` + 출처 표시 |
| 10 | 통합 테스트 | 정확도, 응답시간, 종합 평가 |

다음 주: 프롬프트 엔지니어링 + 메모리 + 최종 완성